# Exp 030 — CoT user-state prompt + Qwen 2.5-7B (DEVSET)

**Pair-test sibling of 029.** Same retrieval (wRRF), same CoT prompt (`response_generation_cot_user_state.txt`), same `max_new_tokens=192`. Only `lm_type` changes: Qwen/Qwen2.5-1.5B-Instruct → **Qwen/Qwen2.5-7B-Instruct**.

**Why this matters**: in local smoke, Qwen 1.5B followed the structured `<user_state>` format on only 2/3 samples — a confound for the ablation. 7B should follow the format ≥95% of the time, isolating the question 'does CoT user-state lift personalization?' from 'does the small model follow CoT?'.

**Risk being tested**: prior-branch Qwen 7B + stock-prompt regressed LLM judge by −0.45 (filler words: 'fantastic', 'perfectly captures'). The new CoT prompt explicitly bans those words AND forces grounded specificity. If 7B+CoT still regresses, the bigger-model penalty is structural (not prompt-fixable) and we revert to the 1.5B path. If 7B+CoT cleanly beats 1.5B+CoT on response quality, the 7B path opens back up for the CoT-equipped pipeline.

## Hardware requirement

- **Runtime → Change runtime type → A100 40GB** (or L4 24GB).
- T4 16GB will OOM (7B bf16 ≈ 14 GB weights + KV cache). Verify with `nvidia-smi` in cell 1.

Wall time on A100: ~25-35 min for 8000 rows at batch 16.

## Read after the run

- Cell 8 prints parser-leak rates + 10 random sample responses. Compare against 029's same cells side-by-side.
- Decision gate: parser-leak <2%, format-follow rate ≥95%, response prose specific (not generic AI-speak).

In [ ]:
# 1) Verify GPU. MUST be A100 or L4 — T4 16GB will OOM on 7B bf16.
!nvidia-smi | head -20
import subprocess
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
gpu_mem_mb = int(subprocess.check_output(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits']).decode().strip())
print(f'\nGPU: {gpu_name}  ({gpu_mem_mb} MB)')
assert gpu_mem_mb >= 22000, f'Need ≥22GB GPU memory for Qwen 7B bf16; got {gpu_mem_mb} MB. Switch runtime to A100 or L4.'
print('GPU memory OK for 7B.')

In [ ]:
# 2) FORCE-FRESH clone — pull latest fresh-model code.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()
!echo -n 'branch:  ' && git rev-parse --abbrev-ref HEAD

In [ ]:
# 2b) Mount Drive + wire persistent caches.
# What this persists across Colab sessions:
#   * HF datasets (talkpl-ai/* — track metadata, user profiles, splits)
#   * BM25 + dense + cf-bpr index caches (otherwise rebuilt 2-3 min/run)
# What this does NOT persist:
#   * HF model weights — Drive read at ~30 MB/s is SLOWER than HF
#     download at ~100 MB/s for 6+ GB models. Models stay on Colab's
#     local cache. (To override, set DRIVE_HF_HOME=True below.)
#
# First run after enabling will COPY existing caches to Drive (slow,
# one time per asset). Subsequent runs are near-instant for datasets +
# zero rebuild for retrieval indices.
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
os.makedirs(f'{DRIVE_BASE}/hf_datasets', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/experiments_cache', exist_ok=True)

# HF datasets cache — env var propagates to !python in cell 5.
os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

# Symlink experiments/cache → Drive so the inference script
# (which uses relative path '../experiments/cache' from
# music-crs-baselines/) sees the persistent version.
import shutil
EXPECTED_CACHE = '/content/recsys2026-lora-tutorial/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

# OPTIONAL: persist HF model weights too (uncomment if you have time on
# the first run for the initial copy and OK with slower model loads).
DRIVE_HF_HOME = False
if DRIVE_HF_HOME:
    os.makedirs(f'{DRIVE_BASE}/hf_models', exist_ok=True)
    os.environ['HF_HOME'] = f'{DRIVE_BASE}/hf_models'
    %env HF_HOME={DRIVE_BASE}/hf_models

print(f'datasets cache  : {os.environ["HF_DATASETS_CACHE"]}')
print(f'experiments dir : {EXPECTED_CACHE} -> {os.readlink(EXPECTED_CACHE)}')
print(f'model cache     : {"DRIVE" if DRIVE_HF_HOME else "local (faster)"}')
!ls -lh {DRIVE_BASE}/

In [ ]:
# 3) Install deps.
# Note on flash-attn: tried pip-installing flash-attn on a prior run; the
# CUDA compile took too long (~10 min) and SDPA on torch ≥2.0 already
# uses memory-efficient attention kernels under the hood. The OOM at
# batch 16 is from KV cache + prefill activations, not attention itself,
# so dropping batch (cell 4) is the cheaper fix.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) Experiment parameters.
TID = '030-cot-user-state-qwen7b-devset'
# 7B + bf16 + sdpa OOMed at batch 16 AND batch 8 on A100 40GB. The
# bottleneck is left-padding to the longest sequence in the batch:
# 8-turn music conversations with metadata expansions can hit 3000-5000
# tokens after the chat template, and the whole batch peaks at the max.
# Batch 4 is the next safe rung. If 4 still OOMs the fix is structural
# (cap tokenizer input length in llama.py — say the word).
BATCH_SIZE = 4
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run devset two-step inference.
# 7B + max_new_tokens=192 (set in yaml) — first run downloads ~15 GB model
# weights from HF (~3 min). Subsequent runs in same Colab session reuse.
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_devset.py \
    --tid {TID} \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate prediction JSON + zip for download.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/devset/{TID}.json'
assert os.path.isfile(SRC), f'prediction not found at {SRC} — did inference fail?'

with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (expected 8000)')
assert len(rows) >= 8000, f'only {len(rows)} rows — partial run; do not score'

stage = f'/content/_stage_{TID}'
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, f'{TID}.json'))
zip_base = f'/content/{TID}'
shutil.make_archive(zip_base, 'zip', stage)
print('wrote', zip_base + '.zip')
!ls -lh {zip_base}.zip

In [ ]:
# 7a) Browser download.
from google.colab import files
files.download(f'/content/{TID}.zip')

In [ ]:
# 7b) Drive backup.
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
dst_dir = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(f'/content/{TID}.zip', dst_dir)
shutil.copy(f'music-crs-baselines/exp/inference/devset/{TID}.json', dst_dir)
print(f'saved to Drive: {dst_dir}')
!ls -lh {dst_dir}

In [ ]:
# 8) Quality probe — sample responses + parser-leak rate.
# 7B should follow the structured format on ≥95% of rows. The parser
# (extract_cot_response in crs_baseline.py) extracts <response>...</response>
# and falls back to stripping <user_state>...</user_state> if no <response>
# tag was emitted. The FINAL parsed text going to Gemini should be near-zero
# leak (no <user_state> / <response> tags, no field names like 'mood:').
import json, random, re

with open(f'music-crs-baselines/exp/inference/devset/{TID}.json') as f:
    rows = json.load(f)

field_leak_re = re.compile(
    r'^(?:mood|intent|energy|sonic_pref|era_pref|familiarity):',
    re.M,
)
tag_leak_re = re.compile(r'<\s*/?\s*(user_state|response)\s*>', re.I)

leak_field, leak_tag, empty = 0, 0, 0
for r in rows:
    resp = (r.get('predicted_response') or '').strip()
    if not resp:
        empty += 1
        continue
    if field_leak_re.search(resp):
        leak_field += 1
    if tag_leak_re.search(resp):
        leak_tag += 1

n = len(rows)
print(f'rows total            : {n}')
print(f'empty responses       : {empty}  ({empty/n:.1%})')
print(f'field-name leak       : {leak_field}  ({leak_field/n:.1%})')
print(f'tag leak              : {leak_tag}  ({leak_tag/n:.1%})')
print()
print('=== 10 random sample responses ===')
random.seed(42)
for i in random.sample(range(n), min(10, n)):
    print(f'\n[{i}] turn {rows[i].get("turn_number")}')
    print(rows[i].get('predicted_response', '')[:400])

# Pass criteria for 7B: leak rates near 0 (parser confidence proven), no
# empty responses. If field/tag leak >2%, the parser missed cases — file an
# update to extract_cot_response.

## After the Colab run, on local M4:

```bash
cd recsys2026
TID=030-cot-user-state-qwen7b-devset
unzip -o ~/Downloads/${TID}.zip -d music-crs-baselines/exp/inference/devset/
source recsys26/bin/activate
python scripts/local_eval.py --tid ${TID} --split dev
pytest tests/test_wave2_integration.py -v
```

## Decision gate (compared to 029 / 1.5B-CoT)

- **7B response quality clearly more specific AND retrieval composite within ±0.005 of 029** → 7B+CoT is the better testbed; promote to a Blind-A candidate (still subject to fresh-model gate policy — no Blind-A ship without explicit user approval).
- **7B response quality similar to 029, no clear lift** → CoT itself is doing the work, stick with cheaper 1.5B for any Blind-A run.
- **7B response quality regresses despite cleaner format** → confirms the prior-branch 7B AI-speak failure mode is structural; 1.5B remains the production LM.